In [175]:
import openseespywin as ops
import opsvis as opsv 
import numpy as np
import ipywidgets as widgets
import os
import matplotlib.pyplot as plt
import math
import opstool as opst
import eurocodepy as ecpy
import time as tt
# import openseespy.postprocessing.Get_Rendering as opsplt

In [176]:
import openseespy.opensees as ops

#Soil Element Properties
elasto_mat_tag = 1
thickness = 1.0
type = "PlaneStrain"

# Soil Material properties
E = 200e6         # Elastic modulus in Pa
nu = 0.3          # Poisson's ratio
rho = 0.0         # Density

# Storage for created nodes and elements
created_nodes = []
created_elements = []
block_registry = {}

def generate_block(start_x, start_y, length, height, node_offset, elem_offset, mat_tag, box_width, box_height, block_name="block"):
    num_x = int(length / box_width)
    num_y = int(height / box_height)

    def block_node_id(i, j):
        return node_offset + j * (num_x + 1) + i + 1

    block_nodes = []
    block_elements = []

    # Create nodes
    for j in range(num_y + 1):
        for i in range(num_x + 1):
            nid = block_node_id(i, j)
            x = start_x + i * box_width
            y = start_y + j * box_height
            ops.node(nid, x, y)
            created_nodes.append((nid, x, y))
            block_nodes.append(nid)

    # Create elements
    eid = elem_offset
    for j in range(num_y):
        for i in range(num_x):
            n1 = block_node_id(i, j)
            n2 = block_node_id(i + 1, j)
            n3 = block_node_id(i + 1, j + 1)
            n4 = block_node_id(i, j + 1)

            ops.element("quad", eid, n1, n2, n3, n4, thickness, type, mat_tag)
            created_elements.append((eid, n1, n2, n3, n4))
            block_elements.append(eid)
            eid += 1

    # Register block
    block_registry[block_name] = {
        "nodes": block_nodes,
        "elements": block_elements,
    }

    return (node_offset + (num_x + 1) * (num_y + 1), elem_offset + num_x * num_y)

# Start model
ops.wipe()
ops.model("Basic", "-ndm", 2, "-ndf", 2)

# Define material
ops.nDMaterial("ElasticIsotropic", elasto_mat_tag, E, nu, rho)

# Define contact material
contact_mat = 2
ops.nDMaterial("ContactMaterial2D", contact_mat, 0.1, 1000.0, 0.0, 0.0)

# Generate one block starting from node 1 and element 1
node_offset = 0
elem_offset = 1
node_offset, elem_offset = generate_block(5, 0.25, 3.0, 7.5, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="OAE_Soil_block")
node_offset, elem_offset = generate_block(8.5, 0.25, 3.0, 7.5, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="F1_Soil_block")


In [177]:
# Fixity definitions
fixXY = [1, 1]
fixXonly = [1, 0]
fixYonly = [0, 1]

# Inputs
start_id = 8
end_id = 106
step = 7

# Generate node list from start to end with step
nodes_to_fix = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in nodes_to_fix:
    ops.fix(nid, *fixXonly)


# Define fixity pattern with step
start_id = 126
end_id = 224

# Generate node list
right_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in right_nodes:
    ops.fix(nid, *fixXonly)


start_id = 1
end_id = 7
step = 1

# Generate node list
bottom_left_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in bottom_left_nodes:
    ops.fix(nid, *fixXY)

start_id = 113
end_id = 119
step = 1

# Generate node list
bottom_right_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in bottom_right_nodes:
    ops.fix(nid, *fixXY)


In [178]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

225


In [179]:
ops.model("Basic", "-ndm", 2, "-ndf", 3)

In [180]:
sheet_pile_node = [
[8.250, 0.000],[8.250, 0.500],[8.250, 1.000],[8.250, 1.500],[8.250, 2.000],[8.250, 2.500],[8.250, 3.000],[8.250, 3.500],
[8.250, 4.000],[8.250, 4.500],[8.250, 5.000],[8.250, 5.500],[8.250, 6.000],[8.250, 6.500],[8.250, 7.000],[8.250, 7.500],[8.250, 8.000],
]

start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(sheet_pile_node):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["sheet_pile_nodes"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [181]:
ops.fix(225, 0, 1, 0)

In [182]:
opst.vis.plotly.plot_model()

In [183]:
# Inputs
lagrange_start_id = 300
num_lagrange_nodes = 16
x_coord = 8.25
y_coord = 0.0

# Output list
lagrange_node_ids = []

# Create nodes
for i in range(num_lagrange_nodes):
    nid = lagrange_start_id + i
    ops.node(nid, x_coord, y_coord)
    created_nodes.append((nid, x_coord, y_coord))
    lagrange_node_ids.append(nid)

# Register block
block_registry["lagrange_nodes"] = {
    "nodes": lagrange_node_ids,
    "elements": []
}

In [184]:
lagrange_node_ids = block_registry["lagrange_nodes"]["nodes"]
print(lagrange_node_ids)

[300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315]


In [185]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

316


In [186]:
master_node_ids = block_registry["sheet_pile_nodes"]["nodes"]
print(master_node_ids)

[225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241]


In [187]:
def generate_node_list(start_id, end_id, step):
    return list(range(start_id, end_id + 1, step))
final_slave_nodes = generate_node_list(7, 112, 7)


In [188]:
master_nodes = master_node_ids
slave_nodes = final_slave_nodes
lagrange_nodes = lagrange_node_ids

contact_elem_ids = []
beam_start_id = 1000
for i in range(len(slave_nodes)):
    tag = beam_start_id + i
    iN = master_nodes[i]
    jN = master_nodes[i + 1]
    sN = slave_nodes[i]
    IN = lagrange_nodes[i]

    ops.element("BeamContact2D", tag, iN, jN, sN, IN, contact_mat, 0.5, 1e-10, 1e-10)
    created_elements.append((tag, iN, jN, sN, IN))
    contact_elem_ids.append(tag)

block_registry["contact_beams_right"] = {
    "nodes": master_nodes + slave_nodes + lagrange_nodes,
    "elements": contact_elem_ids
}

In [189]:
left_slave_nodes = generate_node_list(113, 218, 7)

In [190]:
master_nodes = master_node_ids
slave_nodes = left_slave_nodes
lagrange_nodes = lagrange_node_ids

contact_elem_ids = []
beam_start_id = 1020
for i in range(len(slave_nodes)):
    tag = beam_start_id + i
    iN = master_nodes[i]
    jN = master_nodes[i + 1]
    sN = slave_nodes[i]
    IN = lagrange_nodes[i]

    ops.element("BeamContact2D", tag, iN, jN, sN, IN, contact_mat, 0.5, 1e-10, 1e-10)
    created_elements.append((tag, iN, jN, sN, IN))
    contact_elem_ids.append(tag)

block_registry["contact_beams_left"] = {
    "nodes": master_nodes + slave_nodes + lagrange_nodes,
    "elements": contact_elem_ids
}

In [191]:
opst.vis.plotly.plot_model()

In [192]:
# Inputs
beam_nodes = master_node_ids  # Example node list (ordered start → end)
transFTag = 1
beam_secTag = 1
intTag = 401
Nint = 3
beam_start_id = 3000  # Starting element tag


# Geometry transformation and section definition
ops.geomTransf("Linear", transFTag)
ops.section("Elastic", beam_secTag, 200e6, 0.5, 0.000975)
ops.beamIntegration("Legendre", intTag, beam_secTag, Nint)

# Create elements
beam_elem_ids = []
for i in range(len(beam_nodes) - 1):
    sN = beam_nodes[i]
    eN = beam_nodes[i + 1]
    eid = beam_start_id + i
    ops.element("dispBeamColumn", eid, sN, eN, transFTag, intTag)
    created_elements.append((eid, sN, eN))
    beam_elem_ids.append(eid)

# Register block
block_registry["beam_elements"] = {
    "nodes": beam_nodes,
    "elements": beam_elem_ids
}

In [193]:
opst.vis.plotly.plot_model()